# Day 15: Walk-Forward Modeling

This notebook trains Random Forest and Gradient Boosting regressors using walk-forward validation. Missing feature values are median-imputed inside each fold using the training data only.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

cwd = Path.cwd()
root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(root / 'src'))

from feature_engineering import compute_all_features, event_based_generator, leakage_audit

data_path = root / 'data' / 'processed' / 'Davao_Earthquakes_with_Dist.csv'
soutput_dir = root / 'outputs'
output_dir.mkdir(exist_ok=True)

data_path

## Load Events

In [ ]:
events = pd.read_csv(data_path)
events['province'] = (
    events['location']
    .astype(str)
    .str.extract(r'\(([^)]+)\)')[0]
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
events = events.dropna(subset=['province'])

province_map = {
    'Davao del Norte': 'Davao Del Norte',
    'Davao del Sur': 'Davao Del Sur',
    'Island Garden City of Samal': 'Island Garden City Of Samal',
    'Municipality of Sarangani': 'Municipality Of Sarangani',
}
valid_provinces = [
    'Davao De Oro',
    'Davao Del Norte',
    'Davao Del Sur',
    'Davao Occidental',
    'Davao Oriental',
]

events['province'] = events['province'].str.strip().replace(province_map)
events = events[events['province'].isin(valid_provinces)].copy()

events.shape, events['province'].value_counts()

## Build Or Load Feature Matrix

In [ ]:
x_path = output_dir / 'feature_matrix_X.pkl'
y_path = output_dir / 'target_y.pkl'

if x_path.exists() and y_path.exists():
    X = pd.read_pickle(x_path)
    y = pd.read_pickle(y_path)
else:
    X, y = event_based_generator(
        events_df=events,
        feature_fn=compute_all_features,
        province_col='province',
        datetime_col='datetime',
        magnitude_col='magnitude',
    )
    X.to_pickle(x_path)
    y.to_pickle(y_path)
    X.to_csv(output_dir / 'feature_matrix_X.csv')
    y.to_csv(output_dir / 'target_y.csv')

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Columns:', list(X.columns))
X.isna().sum().sort_values(ascending=False)

## Leakage Audit

In [ ]:
audit = leakage_audit(
    X=X,
    events_df=events,
    feature_fn=compute_all_features,
    province_col='province',
)
mismatches = audit[(~audit['match_full']) | (~audit['match_past'])]
print('Mismatch count:', len(mismatches))
mismatches.head()

## Walk-Forward Splits

In [ ]:
def make_walk_forward_splits(X, min_train_weeks=104, test_weeks=26):
    dates = pd.Index(X.index.get_level_values('forecast_date').unique()).sort_values()
    splits = []
    start = min_train_weeks

    while start < len(dates):
        train_dates = dates[:start]
        test_dates = dates[start:start + test_weeks]
        if len(test_dates) == 0:
            break

        train_mask = X.index.get_level_values('forecast_date').isin(train_dates)
        test_mask = X.index.get_level_values('forecast_date').isin(test_dates)
        splits.append((train_mask, test_mask, train_dates[-1], test_dates[0], test_dates[-1]))
        start += test_weeks

    return splits

splits = make_walk_forward_splits(X)
len(splits), [(s[2], s[3], s[4]) for s in splits]

## Train And Evaluate

In [ ]:
models = {
    'random_forest': RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    'gradient_boosting': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.04,
        max_depth=3,
        min_samples_leaf=5,
        random_state=42,
    ),
}

metrics = []
prediction_frames = []

for fold, (train_mask, test_mask, train_end, test_start, test_end) in enumerate(splits, start=1):
    X_train, X_test = X.loc[train_mask], X.loc[test_mask]
    y_train, y_test = y.loc[train_mask], y.loc[test_mask]

    for model_name, model in models.items():
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', model),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        metrics.append({
            'fold': fold,
            'model': model_name,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'train_rows': len(X_train),
            'test_rows': len(X_test),
            'mae': mean_absolute_error(y_test, preds),
            'rmse': np.sqrt(mean_squared_error(y_test, preds)),
            'r2': r2_score(y_test, preds),
        })

        fold_preds = X_test.reset_index()[['province', 'forecast_date']].copy()
        fold_preds['fold'] = fold
        fold_preds['model'] = model_name
        fold_preds['actual'] = y_test.to_numpy()
        fold_preds['prediction'] = preds
        prediction_frames.append(fold_preds)

metrics_df = pd.DataFrame(metrics)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

metrics_df.to_csv(output_dir / 'walk_forward_metrics.csv', index=False)
predictions_df.to_csv(output_dir / 'walk_forward_predictions.csv', index=False)

metrics_df

## Summary

In [ ]:
summary = (
    metrics_df
    .groupby('model')[['mae', 'rmse', 'r2']]
    .agg(['mean', 'std'])
    .round(4)
)
summary.to_csv(output_dir / 'walk_forward_summary.csv')
summary